# 🛰️ Coordenadas de satélites GPS a partir de efemérides transmitidas

**Passo a passo** para cálculo das coordenadas **X, Y, Z** (ECEF) de cada satélite GPS, usando efemérides transmitidas pelo arquivo de navegação.

**Roteiro**

| Parte | Conteúdo |
|---|---|
| 1 | O que é o RINEX de navegação e como ele é organizado |
| 2 | Leitura do arquivo com o `georinex` (sem renomear colunas) |
| 3 | Visão geral dos parâmetros orbitais transmitidos |
| 4 | Cálculo passo a passo para **um** satélite (equações + código + gráficos) |
| 5 | Referenciais (inercial × fixo à Terra), função completa e validação |
| 6 | Traço no solo (*ground track*) sobre um mapa-múndi |
| 7 | Constelação inteira: **plot 3D (interativo) dos satélites orbitando a Terra** |

> **Dados:** ao executar, o notebook pede o upload de um arquivo RINEX de navegação (`.rnx`, `.nav`, `.YYn` ou `.gz`).


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import gzip, os
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex

import plotly.graph_objects as go
from plotly.colors import qualitative as paleta

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.titlesize": 12, "axes.labelsize": 10, "legend.fontsize": 9,
})
pd.set_option("display.width", 300)
pd.set_option("display.max_columns", 40)

RINEX_PATH = next(iter(uploaded))
print("Arquivo selecionado:", RINEX_PATH)

---
## Parte 1 — O arquivo RINEX de navegação

O **RINEX** (*Receiver INdependent EXchange format*) é o formato padrão de dados GNSS. O arquivo de **navegação** contém as **efemérides transmitidas**: os parâmetros que cada satélite envia na mensagem de navegação para que o receptor calcule a posição dele.

Estrutura do arquivo:

1. **Cabeçalho** (*header*): versão, tipo, parâmetros ionosféricos, diferença GPS–UTC etc. Termina na linha `END OF HEADER`.
2. **Registros**: cada efeméride ocupa **8 linhas** (para GPS). A primeira linha traz o satélite, a época `toc` e os coeficientes do relógio; as 7 seguintes trazem os parâmetros orbitais.

### Mapa dos 8 campos por registro (RINEX 3.x, GPS)

| Linha | Campo 1 | Campo 2 | Campo 3 | Campo 4 |
|---|---|---|---|---|
| 1 | SV + época `toc` | $a_{f0}$ (s) | $a_{f1}$ (s/s) | $a_{f2}$ (s/s²) |
| 2 | IODE | $C_{rs}$ (m) | $\Delta n$ (rad/s) | $M_0$ (rad) |
| 3 | $C_{uc}$ (rad) | $e$ | $C_{us}$ (rad) | $\sqrt{A}$ (√m) |
| 4 | $t_{oe}$ (s da semana GPS) | $C_{ic}$ (rad) | $\Omega_0$ (rad) | $C_{is}$ (rad) |
| 5 | $i_0$ (rad) | $C_{rc}$ (m) | $\omega$ (rad) | $\dot\Omega$ (rad/s) |
| 6 | $\dot i$ (IDOT, rad/s) | códigos L2 | semana GPS | flag L2 P |
| 7 | precisão SV | saúde SV | $T_{GD}$ (s) | IODC |
| 8 | tempo de transmissão | intervalo de ajuste | – | – |

In [ ]:
MU = 3.986005e14           # parâmetro gravitacional da Terra (m³/s²)
WE = 7.2921151467e-5       # velocidade de rotação da Terra (rad/s)
GPS_EPOCH = datetime(1980, 1, 6)

print(f"Arquivo RINEX: {RINEX_PATH}")

### 1.1 Olhando o cabeçalho...

In [ ]:
def abrir(caminho):
    return gzip.open(caminho, "rt") if caminho.endswith(".gz") else open(caminho, "r")

with abrir(RINEX_PATH) as f:
    linhas = f.read().splitlines()

i_fim = next(i for i, l in enumerate(linhas) if "END OF HEADER" in l)
print("=== CABEÇALHO ===")
print("\n".join(linhas[: i_fim + 1]))
print("\n=== PRIMEIRO REGISTRO (8 linhas) ===")
rotulos = ["SV / toc / af0 af1 af2", "IODE  Crs  Δn  M0", "Cuc  e  Cus  √A", "toe  Cic  Ω0  Cis",
           "i0  Crc  ω  Ω̇", "IDOT  L2code  semana  L2P", "acurácia  saúde  TGD  IODC", "t_tx  fit"]
for l, r in zip(linhas[i_fim + 1 : i_fim + 9], rotulos):
    print(f"{l}   <- {r}")

---
## Parte 2 — Lendo o arquivo

In [ ]:
!pip install -q georinex

import georinex as gr

nav = gr.load(RINEX_PATH, use="G").to_dataframe().reset_index().dropna(subset=["Toe"])
nav["toe_abs"] = nav["GPSWeek"] * 604800.0 + nav["Toe"]   # tempo GPS absoluto (s desde 06/01/1980)

print(f"{len(nav)} efemérides lidas  |  {nav['sv'].nunique()} satélites GPS")
nav.head(8)


Cada coluna é um parâmetro da mensagem de navegação:

| Símbolo | Coluna | Símbolo | Coluna | Símbolo | Coluna |
|---|---|---|---|---|---|
| $t_{oc}$ | `time` | $\sqrt{A}$ | `sqrtA` | $C_{uc},\ C_{us}$ | `Cuc`, `Cus` |
| satélite | `sv` | $e$ | `Eccentricity` | $C_{rc},\ C_{rs}$ | `Crc`, `Crs` |
| $a_{f0},a_{f1},a_{f2}$ | `SVclockBias`, `SVclockDrift`, `SVclockDriftRate` | $i_0$ | `Io` | $C_{ic},\ C_{is}$ | `Cic`, `Cis` |
| $\Delta n$ | `DeltaN` | $\Omega_0$ | `Omega0` | $t_{oe}$ (s da semana) | `Toe` |
| $M_0$ | `M0` | $\omega$ | `omega` | semana GPS | `GPSWeek` |
| $\dot\Omega$ | `OmegaDot` | $\dot i$ | `IDOT` | $t_{oe}$ absoluto | `toe_abs` (criada acima) |




In [ ]:
vista = nav[["sv", "time"]].copy()
vista["A_km"] = (nav["sqrtA"] ** 2 / 1e3).round(3)
vista["e"] = nav["Eccentricity"]
for col in ["Io", "Omega0", "omega", "M0"]:
    vista[col + "_deg"] = np.degrees(nav[col]).round(3)
vista.head(6)


---
## Parte 3 — Visão geral dos parâmetros orbitais

Os seis parâmetros **keplerianos** da efeméride descrevem a órbita:

| Parâmetro | Símbolo | Significado geométrico |
|---|---|---|
| $\sqrt{A}$ | semieixo maior | **tamanho** da elipse ($A=(\sqrt A)^2$) |
| $e$ | excentricidade | **forma** da elipse (0 = círculo) |
| $i_0$ | inclinação | ângulo do plano orbital com o equador |
| $\Omega_0$ | ascensão reta do nodo ascendente | **orientação** do plano em torno do eixo Z |
| $\omega$ | argumento do perigeu | orientação da elipse **dentro** do plano |
| $M_0$ | anomalia média em $t_{oe}$ | **posição** do satélite na órbita |

Mais os termos de correção: $\Delta n,\ \dot\Omega,\ \dot i,\ C_{uc},C_{us},C_{rc},C_{rs},C_{ic},C_{is}$.

Vamos visualizá-los para toda a constelação. Os 6 planos orbitais aparecem claramente em $\Omega_0$.

In [ ]:
g = nav.groupby("sv").median(numeric_only=True)
x = np.arange(len(g)); rot = g.index
cont = nav["sv"].value_counts().sort_index()

fig, ax = plt.subplots(2, 3, figsize=(15, 7.5))
ax = ax.ravel()

ax[0].bar(cont.index, cont.values, color="#4c78a8")
ax[0].set_title("Nº de efemérides por satélite"); ax[0].set_ylabel("registros")

ax[1].scatter(x, (g["sqrtA"] ** 2) / 1e3, c="#f58518"); ax[1].axhline(26560, ls="--", c="gray", label="≈26 560 km")
ax[1].set_title("Semieixo maior  A"); ax[1].set_ylabel("km"); ax[1].legend()

ax[2].bar(x, g["Eccentricity"], color="#54a24b"); ax[2].set_title("Excentricidade  e (quase circulares)")

ax[3].scatter(x, np.degrees(g["Io"]), c="#e45756"); ax[3].axhline(55, ls="--", c="gray", label="55° nominal")
ax[3].set_title("Inclinação  i₀"); ax[3].set_ylabel("graus"); ax[3].legend()

ax[4].scatter(x, np.degrees(g["Omega0"]), c=np.degrees(g["Omega0"]), cmap="hsv")
ax[4].set_title("Ω₀  →  6 planos orbitais"); ax[4].set_ylabel("graus")

ax[5].scatter(x, np.degrees(g["M0"]) % 360, c="#b279a2")
ax[5].set_title("M₀ (posição na órbita, varia com toe)"); ax[5].set_ylabel("graus")

for a in ax:
    a.set_xticks(x[::2]); a.set_xticklabels(rot[::2], rotation=90, fontsize=8)
plt.tight_layout(); plt.show()


---
## Parte 4 — Cálculo para **um** satélite

Escolhemos um satélite e uma época de interesse. O código transforma os parâmetros transmitidos em coordenadas **ECEF** (*Earth-Centered, Earth-Fixed*: referencial WGS-84 **fixo à Terra**)

**Fluxo do algoritmo:**

$$
t_k \;\to\; M_k \;\to\; E_k \;\to\; \nu_k \;\to\; \Phi_k \;\to\; (\delta u,\delta r,\delta i) \;\to\; (u_k, r_k, i_k) \;\to\; (x'_k, y'_k) \;\to\; \Omega_k \;\to\; (X,Y,Z)
$$

### Passo 0 — Escolha do satélite, da época e da efeméride

A efeméride correta é a **mais próxima** da época de interesse (a transmitida com $t_{oe}$ mais próximo de $t$). Usamos tempo GPS **absoluto** (segundos desde 06/01/1980) para evitar problemas de virada de semana.

In [ ]:
SAT_DEMO = "G23" if "G23" in nav["sv"].values else nav["sv"].iloc[0]

def escolher_efemeride(df, sv, t_abs):
    """Linha de `nav` (efeméride) do satélite `sv` cujo toe é o mais próximo de t_abs."""
    d = df[df["sv"] == sv]
    return d.loc[(d["toe_abs"] - t_abs).abs().idxmin()]

eph_demo = escolher_efemeride(nav, SAT_DEMO, nav["toe_abs"].median())
t_abs = eph_demo["toe_abs"] + 1800.0           # época de interesse: toe + 30 min

print(f"Satélite: {SAT_DEMO}")
print(f"toe (s da semana) = {eph_demo['Toe']:.0f}   |   semana GPS = {int(eph_demo['GPSWeek'])}")
print(f"Época de interesse t (GPS abs) = {t_abs:.1f} s")
eph_demo[["sqrtA", "Eccentricity", "Io", "Omega0", "omega", "M0", "DeltaN", "OmegaDot", "IDOT",
          "Cuc", "Cus", "Crc", "Crs", "Cic", "Cis"]].to_frame("valor")


### Passo 1 — Tempo desde a época de referência $t_k$

Com tempos em segundos-da-semana, é preciso corrigir a virada de semana: se $t_k>302\,400$ subtrai-se 604 800; se $t_k<-302\,400$ soma-se 604 800. Como usamos tempo absoluto, isso já é automático, mas deixamos a função de correção aqui por clareza.

In [ ]:
def corrige_semana(tk):
    """Correção de virada de semana."""
    return np.where(tk > 302400, tk - 604800, np.where(tk < -302400, tk + 604800, tk))

tk = t_abs - eph_demo["toe_abs"]
print(f"tk = {tk:.1f} s")
print("Teste da correção de semana: t_sow=1000, toe=603000 ->", corrige_semana(1000 - 603000), "s")

### Passo 2 — Movimento médio e anomalia média $M_k$

Semieixo maior e movimento médio **calculado** (Kepler, 3ª lei):

$$A = (\sqrt{A})^2, \qquad n_0 = \sqrt{\frac{\mu}{A^3}}$$

O satélite transmite a correção $\Delta n$ (efeito do achatamento e perturbações):

$$n = n_0 + \Delta n$$

A **anomalia média** cresce linearmente com o tempo — é a "fase" de um satélite fictício que percorre a órbita com velocidade angular constante:

$$M_k = M_0 + n\,t_k$$

In [ ]:
A   = eph_demo["sqrtA"] ** 2
n0  = np.sqrt(MU / A**3)
n   = n0 + eph_demo["DeltaN"]
Mk  = eph_demo["M0"] + n * tk
T   = 2 * np.pi / n

print(f"A   = {A/1e3:.3f} km")
print(f"n0  = {n0:.6e} rad/s   |  Δn = {eph_demo['DeltaN']:.3e} rad/s")
print(f"Período orbital T = {T:.1f} s = {T/3600:.4f} h  (meio dia sideral ≈ 11,967 h)")
print(f"Mk  = {Mk:.6f} rad = {np.degrees(Mk)%360:.3f}°")

### Passo 3 — Equação de Kepler: anomalia excêntrica $E_k$

A anomalia média **não** é o ângulo geométrico real. A relação entre $M$ e a **anomalia excêntrica** $E$ é a **equação de Kepler**:

$$M_k = E_k - e\,\sin E_k$$

Ela é **transcendental** (não tem solução fechada para $E$), então resolvemos **iterativamente** pelo método de Newton–Raphson. Com $f(E)=E-e\sin E-M$ e $f'(E)=1-e\cos E$:

$$E_{j+1} = E_j - \frac{E_j - e\sin E_j - M_k}{1-e\cos E_j},\qquad E_0 = M_k$$

Para a excentricidade dos GPS ($e<0{,}02$) bastam 3–4 iterações.

In [ ]:
def kepler(M, e, tol=1e-13, maxit=30):
    """Resolve M = E - e sin E por Newton-Raphson. Retorna E e o histórico de |ΔE|."""
    M = np.asarray(M, float)
    E = M.copy()
    hist = []
    for _ in range(maxit):
        dE = (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
        E = E - dE
        hist.append(float(np.max(np.abs(dE))))
        if hist[-1] < tol:
            break
    return E, hist

e = eph_demo["Eccentricity"]
Mk_w = (Mk + np.pi) % (2 * np.pi) - np.pi     # normaliza para (-π, π]
Ek, hist = kepler(Mk_w, e)
print(f"e = {e:.6f}")
print(f"Ek = {Ek:.10f} rad   (iterações: {len(hist)})")
print("|ΔE| por iteração:", ["%.2e" % h for h in hist])
print(f"Verificação: Ek - e·sin(Ek) - Mk = {Ek - e*np.sin(Ek) - Mk_w:.2e}")

In [ ]:
# Visualização: Convergência e efeito de "e"
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))

# (a) convergência do Newton para vários e
M_test = 2.0
for ee in [0.01, 0.1, 0.3, 0.6, 0.9]:
    E_ = M_test; erros = []
    for _ in range(8):
        E_ = E_ - (E_ - ee * np.sin(E_) - M_test) / (1 - ee * np.cos(E_))
        erros.append(abs(E_ - ee * np.sin(E_) - M_test))
    ax[0].semilogy(range(1, 9), np.maximum(erros, 1e-17), "o-", label=f"e = {ee}")
ax[0].set_title("(a) Convergência de Newton"); ax[0].set_xlabel("iteração"); ax[0].set_ylabel("|resíduo da equação|"); ax[0].legend()

# (b) E em função de M
Mg = np.linspace(0, 2 * np.pi, 400)
for ee in [0, 0.3, 0.6, 0.9]:
    Eg, _ = kepler(Mg, ee)
    ax[1].plot(np.degrees(Mg), np.degrees(Eg), label=f"e = {ee}")
ax[1].set_title("(b) E(M): quanto maior e, mais 'curvada'"); ax[1].set_xlabel("M (°)"); ax[1].set_ylabel("E (°)"); ax[1].legend()

# (c) diferença E - M
for ee in [0.01, 0.1, 0.3, 0.6]:
    Eg, _ = kepler(Mg, ee)
    ax[2].plot(np.degrees(Mg), np.degrees(Eg - Mg), label=f"e = {ee}")
ax[2].set_title("(c) E − M (amplitude ≈ e rad)"); ax[2].set_xlabel("M (°)"); ax[2].set_ylabel("E − M (°)"); ax[2].legend()
plt.tight_layout(); plt.show()


> * **(a) Convergência** — o eixo vertical é o resíduo da equação, $|E-e\sin E-M|$, em escala log. O método de Newton tem convergência *quadrática*: a cada iteração o número de dígitos corretos aproximadamente dobra. Mesmo com $e=0{,}9$ (muito mais que qualquer GPS) poucas iterações bastam.
> * **(b) $E(M)$** — com $e=0$ a órbita é circular e $E=M$ (reta). Quanto maior $e$, mais a curva se afasta da reta: perto do perigeu o satélite anda mais rápido, e $M$ (que cresce uniformemente) deixa de ser um ângulo geométrico.
> * **(c) $E-M$** — a diferença oscila com amplitude ≈ $e$ rad. Para um GPS com $e\approx0{,}01$ isso é só ~0,6°, mas equivale a $e\,A\approx 265$ km ao longo da órbita: impossível de ignorar.

### Passo 4 — Anomalia verdadeira $\nu_k$

$E$ ainda é um ângulo auxiliar medido a partir do **centro** da elipse. O ângulo que interessa fisicamente é a **anomalia verdadeira** $\nu$: medida a partir do **foco** (onde está o centro da Terra), entre o perigeu e o satélite.

$$\nu_k = \operatorname{atan2}\!\left(\sqrt{1-e^2}\,\sin E_k,\ \cos E_k - e\right)$$

O raio geométrico (distância ao centro da Terra) vale

$$r = A\,(1 - e\cos E_k)$$

In [ ]:
nu_k = np.arctan2(np.sqrt(1 - e**2) * np.sin(Ek), np.cos(Ek) - e)
r_geom = A * (1 - e * np.cos(Ek))
print(f"νk = {nu_k:.8f} rad = {np.degrees(nu_k):.4f}°")
print(f"r (geométrico) = {r_geom/1e3:.3f} km")

### Passo 5 — Argumento de latitude $\Phi_k$ e correções harmônicas

O ângulo do satélite medido a partir do **nodo ascendente** (onde ele cruza o equador subindo) é o **argumento de latitude**:

$$\Phi_k = \nu_k + \omega$$

O modelo kepleriano puro ignora perturbações (achatamento da Terra, Sol, Lua). A efeméride corrige isso com **termos harmônicos de 2ª ordem** (senos e cossenos de $2\Phi_k$):

$$
\begin{aligned}
\delta u_k &= C_{us}\sin 2\Phi_k + C_{uc}\cos 2\Phi_k &&\text{(argumento de latitude)}\\
\delta r_k &= C_{rs}\sin 2\Phi_k + C_{rc}\cos 2\Phi_k &&\text{(raio)}\\
\delta i_k &= C_{is}\sin 2\Phi_k + C_{ic}\cos 2\Phi_k &&\text{(inclinação)}
\end{aligned}
$$

In [ ]:
Phi = nu_k + eph_demo["omega"]
s2, c2 = np.sin(2 * Phi), np.cos(2 * Phi)
du = eph_demo["Cus"] * s2 + eph_demo["Cuc"] * c2
dr = eph_demo["Crs"] * s2 + eph_demo["Crc"] * c2
di = eph_demo["Cis"] * s2 + eph_demo["Cic"] * c2

print(f"Φk = {np.degrees(Phi)%360:.4f}°")
print(f"δu = {du:.3e} rad  (≈ {du*A:.2f} m ao longo da órbita)")
print(f"δr = {dr:.3f} m")
print(f"δi = {di:.3e} rad  (≈ {di*A:.3f} m fora do plano)")

### Passo 6 — Argumento de latitude, raio e inclinação **corrigidos**

$$
\begin{aligned}
u_k &= \Phi_k + \delta u_k\\
r_k &= A\,(1 - e\cos E_k) + \delta r_k\\
i_k &= i_0 + \dot i\,t_k + \delta i_k
\end{aligned}
$$

In [ ]:
u_k = Phi + du
r_k = A * (1 - e * np.cos(Ek)) + dr
i_k = eph_demo["Io"] + eph_demo["IDOT"] * tk + di

print(f"uk = {np.degrees(u_k)%360:.5f}°")
print(f"rk = {r_k/1e3:.4f} km")
print(f"ik = {np.degrees(i_k):.5f}°")

### Passo 7 — Posição no **plano orbital**

No plano da órbita, com eixo $x'$ apontando para o nodo ascendente e a origem no centro da Terra:

$$x'_k = r_k\cos u_k, \qquad y'_k = r_k\sin u_k$$

In [ ]:
xp = r_k * np.cos(u_k)
yp = r_k * np.sin(u_k)
print(f"x' = {xp/1e3:.4f} km\ny' = {yp/1e3:.4f} km   (|r| = {np.hypot(xp, yp)/1e3:.4f} km)")

### Passo 8 — Longitude do nodo ascendente **no referencial fixo à Terra** $\Omega_k$

Até aqui tudo vale no **plano orbital**. Para posicioná-lo em relação à Terra precisamos da longitude do seu nodo ascendente, e é aqui que entra a diferença entre dois referenciais (detalhados na Parte 5):

* **Inercial** (eixos parados em relação às estrelas): o plano orbital praticamente não se move; só precessa devagar com a taxa transmitida $\dot\Omega$ (≈ $-8\times10^{-9}$ rad/s):

$$\Omega_k^{\text{in}} = \Omega_0 + \dot\Omega\,t_k$$

* **Fixo à Terra** (*body-fixed*, ECEF): os eixos **giram junto com o planeta** a $\dot\Omega_e$. Em $t_{sow}=0$ (início da semana GPS) os dois referenciais coincidem, e $\Omega_0$ é medido no referencial inercial "congelado" nesse instante. Desde então a Terra girou $\theta=\dot\Omega_e\,t_{sow}=\dot\Omega_e\,(t_{oe}+t_k)$, e o nodo, visto da Terra, "anda para trás" exatamente esse ângulo:

$$\Omega_k = \Omega_k^{\text{in}} - \dot\Omega_e\,(t_{oe}+t_k) \;=\; \Omega_0 + (\dot\Omega-\dot\Omega_e)\,t_k - \dot\Omega_e\,t_{oe}$$

O termo $-\dot\Omega_e\,t_k$ é a rotação da Terra **durante** o intervalo $t_k$, e o termo $-\dot\Omega_e\,t_{oe}$ é a rotação **acumulada** desde o início da semana até a época de referência $t_{oe}$.


In [ ]:
Omega_k = (eph_demo["Omega0"] + (eph_demo["OmegaDot"] - WE) * tk - WE * eph_demo["Toe"])
print(f"Ω0 = {np.degrees(eph_demo['Omega0']):.4f}°")
print(f"Ωk = {np.degrees(Omega_k):.4f}°  (mod 360 = {np.degrees(Omega_k)%360:.4f}°)")
print(f"Rotação da Terra até toe: {np.degrees(WE*eph_demo['Toe'])%360:.3f}°")

### Passo 9 — Coordenadas ECEF (X, Y, Z)

Agora basta **rotacionar** o plano orbital: primeiro **inclinando-o** por $i_k$ em torno do eixo $x'$ (nodo ascendente) e depois **girando-o** por $\Omega_k$ em torno do eixo Z:

$$
\begin{bmatrix}X\\Y\\Z\end{bmatrix}
= R_3(-\Omega_k)\,R_1(-i_k)
\begin{bmatrix}x'_k\\y'_k\\0\end{bmatrix}
\;\Rightarrow\;
\begin{aligned}
X &= x'_k\cos\Omega_k - y'_k\cos i_k\sin\Omega_k\\
Y &= x'_k\sin\Omega_k + y'_k\cos i_k\cos\Omega_k\\
Z &= y'_k\sin i_k
\end{aligned}
$$

In [ ]:
X = xp * np.cos(Omega_k) - yp * np.cos(i_k) * np.sin(Omega_k)
Y = xp * np.sin(Omega_k) + yp * np.cos(i_k) * np.cos(Omega_k)
Z = yp * np.sin(i_k)

print(f"Satélite {SAT_DEMO}  —  coordenadas ECEF (WGS-84)")
print(f"  X = {X:16.3f} m")
print(f"  Y = {Y:16.3f} m")
print(f"  Z = {Z:16.3f} m")
print(f"  |r| = {np.sqrt(X**2+Y**2+Z**2)/1e3:.4f} km   (deve ser = rk = {r_k/1e3:.4f} km)")

In [ ]:
# Visualização 3D das duas rotações
def anel(ang_u, r_):
    return r_ * np.cos(ang_u), r_ * np.sin(ang_u)

ug = np.linspace(0, 2 * np.pi, 300)
R0 = A / 1e3
xo, yo = anel(ug, R0)               # órbita circular só para o desenho
xs, ys = xp / 1e3, yp / 1e3

def estagio(ang_i, ang_Om):
    def rot(x, y):
        x1, y1, z1 = x, y * np.cos(ang_i), y * np.sin(ang_i)
        return (x1 * np.cos(ang_Om) - y1 * np.sin(ang_Om),
                x1 * np.sin(ang_Om) + y1 * np.cos(ang_Om), z1)
    return rot(xo, yo), rot(np.array([xs]), np.array([ys]))

titulos = ["(a) Plano orbital\n(i=0, Ω=0)", "(b) Inclina por i em torno de x'\n(nodo ascendente)", "(c) Gira por Ω em torno de Z\n→ ECEF"]
estagios = [(0, 0), (i_k, 0), (i_k, Omega_k)]

fig = plt.figure(figsize=(16, 5.5))
L = 32000
th = np.linspace(0, 2 * np.pi, 200)
for k, (ti, (ai, aO)) in enumerate(zip(titulos, estagios)):
    ax = fig.add_subplot(1, 3, k + 1, projection="3d")
    (ox, oy, oz), (sx, sy, sz) = estagio(ai, aO)
    ax.plot(R0 * np.cos(th), R0 * np.sin(th), 0 * th, c="gray", ls="--", lw=1, label="equador")
    ax.plot(ox, oy, oz, "b", lw=2, label="órbita")
    ax.plot(sx, sy, sz, "r*", ms=15)
    nodo = (R0 * np.cos(aO), R0 * np.sin(aO), 0)
    ax.plot([0, nodo[0]], [0, nodo[1]], [0, 0], "g-", lw=2, label="nodo asc.")
    ax.plot([0, sx[0]], [0, sy[0]], [0, sz[0]], "r-", lw=1)
    ax.plot([0, L * 0.8], [0, 0], [0, 0], "k-", lw=0.8); ax.plot([0, 0], [0, L * 0.8], [0, 0], "k-", lw=0.8); ax.plot([0, 0], [0, 0], [0, L * 0.8], "k-", lw=0.8)
    ax.text(L * 0.85, 0, 0, "X"); ax.text(0, L * 0.85, 0, "Y"); ax.text(0, 0, L * 0.85, "Z")
    ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_zlim(-L, L); ax.set_box_aspect((1, 1, 1))
    ax.set_title(ti, fontsize=10); ax.view_init(22, 35)
    if k == 0: ax.legend(loc="upper left", fontsize=7)
plt.tight_layout(); plt.show()

> * **(a) Plano orbital** — tudo acontece num único plano: o eixo $x'$ (verde) aponta para o nodo ascendente e o satélite está em $(x'_k,\,y'_k)$. Equivale a $i=0$ e $\Omega=0$.
> * **(b) Inclinação** — giramos o plano por $i_k$ em torno de $x'$ (a linha dos nodos). A órbita deixa de coincidir com o equador (tracejado cinza) e passa a subir até ≈ 55° de latitude.
> * **(c) Rotação em torno de Z** — giramos tudo por $\Omega_k$, apontando o nodo ascendente para a longitude correta **no referencial fixo à Terra**. A posição final da estrela é o $(X,Y,Z)$ calculado.

---
## Parte 5 — Função completa

Agora reunimos **todos os passos** em uma função para todos os satélites

A função aceita:

* `referencial="ecef"` (padrão) — coordenadas fixas à Terra, o que um receptor usaria;
* `referencial="inercial"` — sem descontar $\theta$: a órbita é uma elipse **fechada**, ideal para **desenhar órbitas**.


In [ ]:
def posicao_satelite(eph, t_abs, referencial="ecef", detalhes=False):
    """
    Posição do satélite GPS pela efeméride transmitida (IS-GPS-200, Tab. 20-IV).
    eph         : linha do DataFrame `nav` (pd.Series; colunas com os nomes do georinex)
    t_abs       : tempo(s) GPS absoluto(s) (s desde 06/01/1980)
    referencial : "ecef" (fixo à Terra, padrão) ou "inercial" (sem a rotação da Terra)
    Retorna X, Y, Z (m) — ou um dicionário com todas as etapas se detalhes=True.
    """
    if referencial not in ("ecef", "inercial"):
        raise ValueError('referencial deve ser "ecef" ou "inercial"')
    t_abs = np.atleast_1d(np.asarray(t_abs, float))
    tk = t_abs - eph["toe_abs"]
    e = eph["Eccentricity"]
    A = eph["sqrtA"] ** 2
    n = np.sqrt(MU / A**3) + eph["DeltaN"]
    M = eph["M0"] + n * tk
    E, _ = kepler((M + np.pi) % (2 * np.pi) - np.pi, e)        # mesma função da Parte 4
    nu = np.arctan2(np.sqrt(1 - e**2) * np.sin(E), np.cos(E) - e)
    Phi = nu + eph["omega"]
    s2, c2 = np.sin(2 * Phi), np.cos(2 * Phi)
    u = Phi + eph["Cus"] * s2 + eph["Cuc"] * c2
    r = A * (1 - e * np.cos(E)) + eph["Crs"] * s2 + eph["Crc"] * c2
    i = eph["Io"] + eph["IDOT"] * tk + eph["Cis"] * s2 + eph["Cic"] * c2
    xp, yp = r * np.cos(u), r * np.sin(u)
    Om = eph["Omega0"] + eph["OmegaDot"] * tk                  # nodo no referencial inercial
    if referencial == "ecef":
        Om = Om - WE * (eph["Toe"] + tk)                       # desconta a rotação da Terra: θ = Ωe·t_sow
    X = xp * np.cos(Om) - yp * np.cos(i) * np.sin(Om)
    Y = xp * np.sin(Om) + yp * np.cos(i) * np.cos(Om)
    Z = yp * np.sin(i)
    if detalhes:
        return dict(tk=tk, M=M, E=E, nu=nu, Phi=Phi, u=u, r=r, i=i, Omega=Om, xp=xp, yp=yp, X=X, Y=Y, Z=Z)
    return X, Y, Z


### 5.1 Sanidade física

Um satélite ao longo de ~1 órbita: o raio, a altitude e a velocidade precisam ter ordens de grandeza coerentes com o GPS.

In [ ]:
# Sanidade física: raio ~ 26 560 km, altitude ~ 20 200 km, velocidade inercial ~ 3,87 km/s (no ECEF ela oscila)
tt = t_abs + np.arange(0, 3600 * 12, 30.0)
Xs, Ys, Zs = posicao_satelite(eph_demo, tt)
r_ = np.sqrt(Xs**2 + Ys**2 + Zs**2)
v = np.sqrt(np.diff(Xs)**2 + np.diff(Ys)**2 + np.diff(Zs)**2) / 30

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
h = (tt - tt[0]) / 3600
ax[0].plot(h, r_ / 1e3); ax[0].set_title("|r| ECEF (km)"); ax[0].set_xlabel("horas")
ax[1].plot(h, (r_ - 6378137) / 1e3, c="#f58518"); ax[1].set_title("Altitude aprox. (km)"); ax[1].set_xlabel("horas")
ax[2].plot(h[1:], v / 1e3, c="#54a24b"); ax[2].set_title("Velocidade no ECEF (km/s)"); ax[2].set_xlabel("horas")
plt.tight_layout(); plt.show()

> * **$|r|$** — a distância ao centro da Terra fica em torno de 26 560 km e varia pouco, no máximo algumas centenas de km (≈ $2\,e\,A$ entre perigeu e apogeu): órbita quase circular.
> * **Altitude** — $|r|$ menos o raio equatorial (6 378 km): ≈ 20 200 km acima da superfície.
> * **Velocidade no ECEF** — no referencial inercial seria praticamente constante, ≈ $\sqrt{\mu/A}\approx 3{,}87$ km/s. No ECEF ela **oscila**: a "velocidade de arrasto" do referencial, $\dot\Omega_e|r|\approx 1{,}9$ km/s, se combina *vetorialmente* com a do satélite, e o módulo resultante muda conforme a posição na órbita.

---
## Parte 6 — Traço no solo (*ground track*)

Convertendo X, Y, Z (ECEF) em **latitude/longitude** geodésicas (elipsoide WGS-84) vemos por onde o satélite passa sobre a superfície. Como o período é ≈ meio dia sideral, o traço **repete-se a cada ~2 voltas por dia sideral** (ressonância 2:1).

Conversão (iterativa):

$$\lambda = \operatorname{atan2}(Y,X),\qquad p=\sqrt{X^2+Y^2},\qquad \varphi_{j+1}=\operatorname{atan2}\!\left(Z,\ p\left(1-\frac{e_E^2 N_j}{N_j+h_j}\right)\right)$$

com $N=a/\sqrt{1-e_E^2\sin^2\varphi}$ e $h=p/\cos\varphi-N$.

Os traços são desenhados **sobre um mapa-múndi** (costas e países), de forma interativa: dá para dar zoom e passar o mouse para ler latitude e longitude.


In [ ]:
def ecef_para_geodesicas(X, Y, Z):
    a, f = 6378137.0, 1 / 298.257223563
    e2 = f * (2 - f)
    lon = np.arctan2(Y, X)
    p = np.hypot(X, Y)
    lat = np.arctan2(Z, p * (1 - e2))
    for _ in range(6):
        N = a / np.sqrt(1 - e2 * np.sin(lat) ** 2)
        h = p / np.cos(lat) - N
        lat = np.arctan2(Z, p * (1 - e2 * N / (N + h)))
    return np.degrees(lat), np.degrees(lon), h

def quebra_longitude(lon, lat):
    """Insere NaN onde a longitude 'pula' de +180 para -180 (evita linha cruzando o mapa)."""
    lon = lon.copy(); lat = lat.copy()
    salto = np.where(np.abs(np.diff(lon)) > 180)[0]
    lon[salto] = np.nan; lat[salto] = np.nan
    return lon, lat

sats_gt = [s for s in ["G01", "G05", "G09", "G13", "G17", "G21"] if s in nav["sv"].values] or list(nav["sv"].unique()[:4])
t0 = nav["toe_abs"].median()
tt = t0 + np.arange(0, 12 * 3600, 60.0)

fig = go.Figure()
for k, s in enumerate(sats_gt):
    Xs, Ys, Zs = posicao_satelite(escolher_efemeride(nav, s, t0), tt)        # ECEF (padrão)
    lat, lon, _ = ecef_para_geodesicas(Xs, Ys, Zs)
    lo, la = quebra_longitude(lon, lat)
    cor = paleta.Plotly[k % len(paleta.Plotly)]
    fig.add_trace(go.Scattergeo(lon=lo, lat=la, mode="lines", name=s, legendgroup=s,
                                line=dict(width=2.5, color=cor)))
    fig.add_trace(go.Scattergeo(lon=[lon[0]], lat=[lat[0]], mode="markers", legendgroup=s, showlegend=False,
                                marker=dict(size=9, color=cor, line=dict(width=1, color="black")),
                                hovertemplate=f"{s} (início)<extra></extra>"))

# paralelos de ±55° (inclinação): o satélite nunca passa daí
for j, lat_lim in enumerate((55, -55)):
    fig.add_trace(go.Scattergeo(lon=np.linspace(-180, 180, 181), lat=np.full(181, lat_lim), mode="lines",
                                name="±55° (inclinação)", legendgroup="incl", showlegend=(j == 0),
                                line=dict(width=1, dash="dash", color="gray"), hoverinfo="skip"))

fig.update_geos(                                   # <-- o mapa de fundo
    projection_type="equirectangular",
    showland=True, landcolor="#e8efd9",
    showocean=True, oceancolor="#cfe3f2",
    showcoastlines=True, coastlinecolor="#555",
    showcountries=True, countrycolor="#aaa",
    lonaxis=dict(range=[-180, 180], showgrid=True, dtick=60),
    lataxis=dict(range=[-90, 90], showgrid=True, dtick=30),
)
fig.update_layout(
    title="Traço no solo de satélites GPS — 12 h  (● = início)",
    height=560, margin=dict(l=0, r=0, t=50, b=0),
    legend=dict(orientation="h", y=-0.02),
)
fig.show()


> * **Forma de senoide** — o satélite sobe e desce em latitude até um máximo ≈ $i_0\approx55^\circ$; ao cruzar o equador ele está nos nodos.
> * **Deslocamento para leste** — em 12 h (≈ 1 órbita) o traço avança só ~180° de longitude: o satélite dá uma volta completa (360°), mas a Terra gira ~180° sob ele no mesmo tempo.
> * **Repetição** — após ~2 órbitas (1 dia sideral) a Terra volta à mesma orientação e o traço se repete (ressonância 2:1).
> * **Referencial** — este é o ECEF, o referencial *body-fixed* da Parte 5. No referencial inercial, sem a rotação $\dot\Omega_e$, cada órbita seria projetada sempre sobre o mesmo círculo máximo.

## Parte 7 — A constelação GPS inteira orbitando a Terra (plot 3D)


Para cada satélite:

1. escolhemos a efeméride **mais próxima** da época $t_{ref}$;
2. calculamos a posição **atual** (pontos), no mesmo referencial das órbitas;
3. desenhamos a **órbita completa** no referencial inercial (elipses fechadas), amostrando um período inteiro com `posicao_satelite(..., referencial="inercial")`.

A Terra é desenhada como esfera com malha de paralelos e meridianos. Observe os **6 planos orbitais** (uma cor por plano), cada um inclinado ≈ 55° em relação ao equador.


In [ ]:
import numpy as np
import plotly.graph_objects as go
from matplotlib.colors import to_hex
from datetime import timedelta

def cor_do_plano(ex):
    """Cor pelo Ω0: satélites do mesmo plano orbital ficam com a mesma cor."""
    return plt.cm.hsv((ex["Omega0"] % (2 * np.pi)) / (2 * np.pi))

def constelacao(nav, t_ref, n=300):
    """Para cada satélite: órbita completa (inercial, km), posição em t_ref (km) e cor."""
    info = []
    for s in sorted(nav["sv"].unique()):
        ex = escolher_efemeride(nav, s, t_ref)
        A_ = ex["sqrtA"] ** 2
        T_ = 2 * np.pi / (np.sqrt(MU / A_**3) + ex["DeltaN"])
        orb = np.array(posicao_satelite(ex, t_ref + np.linspace(0, T_, n), referencial="inercial")) / 1e3
        info.append(dict(sv=s, orbita=orb, pos=orb[:, 0], cor=cor_do_plano(ex)))
    return info

def terra_plotly(R=6378.137, n=60):
    """Esfera da Terra (superfície azul) para o plotly."""
    u, v = np.meshgrid(np.linspace(0, 2 * np.pi, n), np.linspace(0, np.pi, n // 2))
    return go.Surface(x=R * np.cos(u) * np.sin(v), y=R * np.sin(u) * np.sin(v), z=R * np.cos(v),
                      colorscale=[[0, "#9ecae1"], [1, "#9ecae1"]], showscale=False,
                      opacity=0.85, hoverinfo="skip")

# --- Execução e Plotagem Plotly ---
t_ref = nav["toe_abs"].median()
info = constelacao(nav, t_ref)
L = 30000
t_dt = GPS_EPOCH + timedelta(seconds=float(t_ref))

fig = go.Figure(terra_plotly())

# Equador e eixos X, Y, Z
th = np.linspace(0, 2 * np.pi, 200)
fig.add_trace(go.Scatter3d(x=6378 * np.cos(th), y=6378 * np.sin(th), z=0 * th, mode="lines",
                           line=dict(color="#08306b", width=4), hoverinfo="skip"))
for nome, vet in {"X": (1, 0, 0), "Y": (0, 1, 0), "Z": (0, 0, 1)}.items():
    fim = np.array(vet) * 1.1 * L
    fig.add_trace(go.Scatter3d(x=[0, fim[0]], y=[0, fim[1]], z=[0, fim[2]], mode="lines+text",
                               text=["", nome], textposition="top center",
                               line=dict(color="black", width=3), hoverinfo="skip"))

# Órbitas (linhas) e posições atuais (marcadores)
for c in info:
    cor = to_hex(c["cor"])
    fig.add_trace(go.Scatter3d(x=c["orbita"][0], y=c["orbita"][1], z=c["orbita"][2], mode="lines",
                               line=dict(color=cor, width=3), name=c["sv"], hoverinfo="name"))
    fig.add_trace(go.Scatter3d(x=[c["pos"][0]], y=[c["pos"][1]], z=[c["pos"][2]], mode="markers+text",
                               text=[c["sv"]], textposition="top center",
                               marker=dict(size=5, color=cor, line=dict(color="black", width=1)),
                               hovertemplate="<b>%{text}</b><br>X = %{x:,.0f} km<br>Y = %{y:,.0f} km<br>Z = %{z:,.0f} km<extra></extra>"))

fig.update_layout(
    title=f"Constelação GPS — {len(info)} satélites | época {t_dt:%Y-%m-%d %H:%M:%S} (tempo GPS)",
    height=750, margin=dict(l=0, r=0, t=50, b=0), showlegend=False,
    scene=dict(aspectmode="cube",
               xaxis=dict(range=[-L, L], title="X (km)"),
               yaxis=dict(range=[-L, L], title="Y (km)"),
               zaxis=dict(range=[-L, L], title="Z (km)"),
               camera=dict(eye=dict(x=1.5, y=1.5, z=0.9))),
)

fig.show()

### Tabela final: coordenadas XYZ (ECEF) de todos os satélites

Resultado final do notebook: para a época $t_{ref}$, a posição de cada satélite no referencial **ECEF (WGS-84)**, já com latitude/longitude/altitude.

In [ ]:
sats = sorted(nav["sv"].unique())
linhas = []
for s in sats:
    ex = escolher_efemeride(nav, s, t_ref)
    Xs_, Ys_, Zs_ = (c[0] for c in posicao_satelite(ex, t_ref, referencial="ecef"))
    lat, lon, h = ecef_para_geodesicas(Xs_, Ys_, Zs_)
    linhas.append(dict(sv=s, Toe=ex["Toe"], X_m=Xs_, Y_m=Ys_, Z_m=Zs_, lat_deg=float(lat), lon_deg=float(lon), alt_km=float(h) / 1e3))
tabela = pd.DataFrame(linhas).set_index("sv")
tabela.round({"X_m": 3, "Y_m": 3, "Z_m": 3, "lat_deg": 3, "lon_deg": 3, "alt_km": 2})


---
## Resumo do algoritmo

| # | Grandeza | Equação |
|---|---|---|
| 1 | Tempo desde a referência | $t_k = t - t_{oe}$ |
| 2 | Semieixo e mov. médio | $A=(\sqrt A)^2,\ n=\sqrt{\mu/A^3}+\Delta n$ |
| 3 | Anomalia média | $M_k = M_0 + n\,t_k$ |
| 4 | Kepler | $M_k = E_k - e\sin E_k$ (iterativo) |
| 5 | Anomalia verdadeira | $\nu_k=\operatorname{atan2}(\sqrt{1-e^2}\sin E_k,\ \cos E_k-e)$ |
| 6 | Arg. de latitude | $\Phi_k=\nu_k+\omega$ |
| 7 | Correções | $\delta u,\delta r,\delta i$ (harmônicos de $2\Phi_k$) |
| 8 | Valores corrigidos | $u_k,\ r_k,\ i_k$ |
| 9 | Plano orbital | $x'=r\cos u,\ y'=r\sin u$ |
| 10 | Nodo no ECEF | $\Omega_k=\Omega_0+(\dot\Omega-\dot\Omega_e)t_k-\dot\Omega_e t_{oe}$ |
| 11 | ECEF | $X=x'\cos\Omega-y'\cos i\sin\Omega,\ Y=x'\sin\Omega+y'\cos i\cos\Omega,\ Z=y'\sin i$ |
| 12 | Inercial → ECEF (*body-fixed*) | $\mathbf r_{\text{ECEF}}=R_3(\theta)\,\mathbf r_{\text{in}},\ \theta=\dot\Omega_e\,t_{sow}$ |

### Próximos passos.....
* **Comparar com SP3** (órbita precisa) para medir o erro da efeméride transmitida (~1 m).

**Referência:** IS-GPS-200 (Interface Specification), *Navstar GPS Space Segment/Navigation User Interfaces*, Tabela 20-IV.